In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

In [3]:
sentences = [
    "I love this product",
    "This movie made me smile",
    "Service was friendly and quick",
    "Today felt bright and happy",
    "This is the best day",
    "Absolutely fantastic experience",
    "I enjoyed every single moment",
    "Great job, well done",
    "The food tasted delicious",
    "Totally recommend to everyone",
    "Very satisfied with results",
    "This worked better than expected",
    "Amazing quality and value",
    "Such a pleasant surprise",
    "I feel positive about this",
    "I hate this product",
    "This movie bored me",
    "Service was rude and slow",
    "Today was cold and lonely",
    "This is the worst day",
    "Terrible experience overall",
    "I regret buying this",
    "Very disappointed with results",
    "The food tasted awful",
    "Do not recomment this",
    "It broke after one use",
    "Not worth the money",
    "Utterly frustating and annoying",
    "I feel negative about this",
    "Such a waste of time"
]

# Creating labels: First 15 sentences → 1 → Positive, Last 15 sentences  → 0 → Negative
labels = [1]*15 + [0]*15
# Convert the Python list into a NumPy array
labels = np.array(labels)

In [10]:
# Stating that we want to work with a vocabulary size of up to 200 words (enough for 30 sentences)
vocab_size = 200

# Sentence Tokenizer (it will assign numeric ids to every word)
tok = Tokenizer(num_words=vocab_size, oov_token="<OOV>")  # oov -> out of vocabulary tokens
# Learn the vocabulary from your sentences eg. learn the id of each word
tok.fit_on_texts(sentences)

# to generate sequence form numeric ids of words
seqs = tok.texts_to_sequences(sentences)

# finding max length from the sequence generated so that we can apply padding to make length of every sentence equal
max_len = max(len(s) for s in seqs)

X = pad_sequences(seqs, maxlen=max_len, padding="post")  # making every sequence of length = max_len
y = labels  # output labels

In [5]:
 X[0]

array([ 3, 25,  2,  7,  0], dtype=int32)

## RNN MODEL

In [6]:
embed_dim = 16
rnn_units = 8

In [11]:
inp = Input(shape= (max_len,), dtype = "int32", name = "input")  # input is the name of the i/p layer
x = Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero = True, name = "embed")(inp)  # mask_zero is similar to padding but it's for the output vector
rnn = SimpleRNN(units=rnn_units, return_sequences=False, return_state= False, name= "simple_rnn")
x_last = rnn(x)
out = Dense(1, activation="sigmoid", name = "out")(x_last)

model = Model(inputs = inp, outputs = out)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │      3,200 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 5)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,409 (13.32 KB)

 Trainable params: 3,409 (13.32 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.fit(X, y, epochs=25, batch_size=8, verbose=1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.4667 - loss: 0.6932
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5667 - loss: 0.6795
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7000 - loss: 0.6668
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7667 - loss: 0.6548
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8333 - loss: 0.6423
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9000 - loss: 0.6294
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9000 - loss: 0.6159
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9000 - loss: 0.6010
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9667 - loss: 0.5849 
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9667 - loss: 0.5679
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9667 - loss: 0.5487
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9667 - loss: 0.5283
